# CNN-Based Malaria Parasite Recognition in Human Blood Smear Images


---
## Member 2: Pipeline Implementation

**Goal:** Data quality checks, common transform policy, and DataLoader creation for all 3 datasets.  
All 9 baseline training runs (ResNet-18, VGG-16, MobileNet-V2 × miracle9to9, orvile, malaria) consume `dataloaders_registry` directly.


### Step 1 - Imports & Constants


In [ ]:
from pathlib import Path
import pandas as pd
import torch
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

DATA_ROOT        = Path("../data/raw")
IMAGE_INPUT_SIZE = 224
BATCH_SIZE       = 32
NUM_WORKERS      = 0 # No multiprocessing in Dataloader
DATASET_NAMES    = ["miracle9to9", "orvile", "malaria"]
IMAGE_EXTENSIONS = {".jpg", ".png"}

# ImageNet statistics  applies to ResNet-18, VGG-16, and MobileNet-V2
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


### Step 2 - Data Quality Checks

Inventory every image under `data/raw/`, flag zero-byte (broken) files, and report class distribution and imbalance ratios per dataset/split.


In [14]:
all_image_paths = pd.Series([
    path for path in DATA_ROOT.rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
])

image_inventory_df = pd.DataFrame({
    "dataset":   all_image_paths.map(lambda path: path.relative_to(DATA_ROOT).parts[0]),
    "split":     all_image_paths.map(lambda path: path.relative_to(DATA_ROOT).parts[1]),
    "class_raw": all_image_paths.map(lambda path: path.parent.name),
    "file_size": all_image_paths.map(lambda path: path.stat().st_size),
    "filepath":  all_image_paths,
})

def is_valid_image(path):
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False

image_inventory_df["is_valid"] = all_image_paths.map(is_valid_image)

broken_files_df = image_inventory_df[~image_inventory_df["is_valid"]]
print(f"Broken/unreadable files found: {len(broken_files_df)}")
if not broken_files_df.empty:
    with open("broken_files_report.txt", "w") as report_file:
        report_file.write("Broken/unreadable files:\n")
        report_file.write(broken_files_df[["dataset", "split", "class_raw", "filepath"]].to_string(index=False))
        report_file.write("\n")
    print(f"\nBroken files report saved to 'broken_files_report.txt' with {len(broken_files_df)} broken files.\n")

valid_images_inventory_df = image_inventory_df[image_inventory_df["is_valid"]]
valid_images_count = len(valid_images_inventory_df)
print(f"\nTotal valid images: {valid_images_count}") 

class_distribution_df = (
    valid_images_inventory_df
    .groupby(["dataset", "split", "class_raw"])
    .size()
    .reset_index(name="image_count")
)
print("\nClass distribution:")
print(class_distribution_df.to_string(index=False))

split_totals_df = (
    valid_images_inventory_df
    .groupby(["dataset", "split"])
    .size()
    .reset_index(name="total_images")
)
print("\nSplit totals:")
print(split_totals_df.to_string(index=False))

imbalance_df = (
    class_distribution_df
    .groupby(["dataset", "split"])["image_count"]
    .agg(max_count="max", min_count="min")
    .assign(imbalance_ratio=lambda dataframe: (dataframe["max_count"] / dataframe["min_count"]).round(2))
    .reset_index()
)
print("\nClass imbalance ratios (max_class_count / min_class_count):")
print(imbalance_df.to_string(index=False))


Broken/unreadable files found: 1532

Broken files report saved to 'broken_files_report.txt' with 1532 broken files.


Total valid images: 43390

Class distribution:
    dataset split   class_raw  image_count
miracle9to9  test Parasitized         7952
miracle9to9  test  Uninfected         7880
miracle9to9 train Parasitized        11713
miracle9to9 train  Uninfected        11712
miracle9to9   val Parasitized         2066
miracle9to9   val  Uninfected         2067

Split totals:
    dataset split  total_images
miracle9to9  test         15832
miracle9to9 train         23425
miracle9to9   val          4133

Class imbalance ratios (max_class_count / min_class_count):
    dataset split  max_count  min_count  imbalance_ratio
miracle9to9  test       7952       7880             1.01
miracle9to9 train      11713      11712             1.00
miracle9to9   val       2067       2066             1.00


### Step 3 - Transform Policy

One shared policy for all 9 baseline training runs.  
- `training_transforms` : resize → geometric augmentation → color jitter → normalize  
- `evaluation_transforms` : resize → normalize only (no augmentation on val/test)


In [15]:
training_transforms = transforms.Compose([
    transforms.Resize((IMAGE_INPUT_SIZE, IMAGE_INPUT_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

evaluation_transforms = transforms.Compose([
    transforms.Resize((IMAGE_INPUT_SIZE, IMAGE_INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print("training_transforms :")
print(training_transforms)
print("\nevaluation_transforms :")
print(evaluation_transforms)


training_transforms :
Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    RandomHorizontalFlip(p=0.5)
    RandomVerticalFlip(p=0.5)
    RandomRotation(degrees=[-15.0, 15.0], interpolation=nearest, expand=False, fill=0)
    ColorJitter(brightness=(0.8, 1.2), contrast=(0.8, 1.2), saturation=(0.8, 1.2), hue=None)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

evaluation_transforms :
Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


### Step 4 - MalariaDataset

Custom `Dataset` that works for all three datasets.  
Accepts a pre-filtered `split_inventory_df` slice from `valid_images_inventory_df` (Step 2), so only validated images are ever loaded.  
Handles `orvile`'s duplicate training folders (`"gametocyte 2"` → `"gametocyte"`) by stripping the trailing numeric suffix via vectorized regex on class folder names.


In [16]:
class MalariaDataset(Dataset):
    def __init__(self, split_inventory_df, transform=None):
        normalized_class_names = split_inventory_df["class_raw"].str.replace(r"\s+\d+$", "", regex=True)

        unique_classes   = sorted(normalized_class_names.unique())
        class_to_index   = {class_name: index for index, class_name in enumerate(unique_classes)}

        self.image_paths  = split_inventory_df["filepath"].to_numpy()
        self.labels       = normalized_class_names.map(class_to_index).to_numpy()
        self.class_names  = unique_classes
        self.class_to_idx = class_to_index
        self.transform    = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        image = Image.open(self.image_paths[index]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, int(self.labels[index])


### Step 5 - Datasets & DataLoaders

`datasets_registry[dataset_name][split]` → `MalariaDataset`  
`dataloaders_registry[dataset_name][split]` → `DataLoader` ready for training  

All 9 scratch models and 2 transfer learning models consume `dataloaders_registry`.


In [17]:
SPLIT_TRANSFORMS = {"train": training_transforms, "val": evaluation_transforms, "test": evaluation_transforms}

datasets_registry = {
    dataset_name: {
        split: MalariaDataset(
            valid_images_inventory_df[
                (valid_images_inventory_df["dataset"] == dataset_name) &
                (valid_images_inventory_df["split"]   == split)
            ].reset_index(drop=True),
            transform=split_transform,
        )
        for split, split_transform in SPLIT_TRANSFORMS.items()
    }
    for dataset_name in DATASET_NAMES
}

dataloaders_registry = {
    dataset_name: {
        split: DataLoader(
            dataset,
            batch_size  = BATCH_SIZE,
            shuffle     = (split == "train"),
            num_workers = NUM_WORKERS,
            pin_memory  = torch.cuda.is_available(),
            drop_last   = (split == "train"),
        )
        for split, dataset in split_datasets.items()
        if len(dataset) > 0
    }
    for dataset_name, split_datasets in datasets_registry.items()
}

for dataset_name, split_datasets in datasets_registry.items():
    for split_name, dataset in split_datasets.items():
        if len(dataset) == 0:
            print(f"{dataset_name:15s} | {split_name:5s} | SKIPPED since 0 valid images")
            continue
        num_batches = len(dataloaders_registry[dataset_name][split_name])
        print(f"{dataset_name:15s} | {split_name:5s} | {len(dataset):5d} images | {num_batches:3d} batches | classes: {dataset.class_names}")


miracle9to9     | train | 23425 images | 732 batches | classes: ['Parasitized', 'Uninfected']
miracle9to9     | val   |  4133 images | 130 batches | classes: ['Parasitized', 'Uninfected']
miracle9to9     | test  | 15832 images | 495 batches | classes: ['Parasitized', 'Uninfected']
orvile          | train | SKIPPED since 0 valid images
orvile          | val   | SKIPPED since 0 valid images
orvile          | test  | SKIPPED since 0 valid images
malaria         | train | SKIPPED since 0 valid images
malaria         | val   | SKIPPED since 0 valid images
malaria         | test  | SKIPPED since 0 valid images
